In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import random

# ====================== 1. 数据路径配置 ======================
BASE_DIR = "./data"  
TRAIN_DIR = os.path.join(BASE_DIR, "TRAIN")
VALIDATION_DIR = os.path.join(BASE_DIR, "VALIDATION")

IMAGE_SIZE = (224, 224)  
BATCH_SIZE = 32
NUM_CLASSES = 5  # left/oob/prone/right/supine
SEED = 42  # 固定随机种子，确保可复现性

# ====================== 2. 数据processing ======================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True, 
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED,
)

validation_generator = validation_datagen.flow_from_directory(
    VALIDATION_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    seed=SEED,
)

# ====================== 3. 模型构建函数 ======================
# MobileNetV2模型
def build_mobilenet_model():
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(*IMAGE_SIZE, 3)
    )
    base_model.trainable = False  # 冻结基础模型
    
    inputs = Input(shape=(*IMAGE_SIZE, 3))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)  # 全连接层提取特征
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)
    
    return Model(inputs=inputs, outputs=outputs)

# ResNet50模型（对比模型）
def build_resnet_model():
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(*IMAGE_SIZE, 3)
    )
    base_model.trainable = False  # 冻结基础模型（可尝试后期解冻微调）
    
    inputs = Input(shape=(*IMAGE_SIZE, 3))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)  # 调整全连接层参数（与MobileNet区分）
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)
    
    return Model(inputs=inputs, outputs=outputs)

# ====================== 4. 模型训练 ======================
# 初始化两个模型
model_mobilenet = build_mobilenet_model()
model_resnet = build_resnet_model()

# 编译模型（统一优化器和损失函数，确保公平对比）
for model in [model_mobilenet, model_resnet]:
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

# 训练MobileNetV2
history_mobilenet = model_mobilenet.fit(
    train_generator,
    epochs=10,  
    validation_data=validation_generator,
    verbose=1,
    shuffle=True,
)

# 训练ResNet50
history_resnet = model_resnet.fit(
    train_generator,
    epochs=10,  
    validation_data=validation_generator,
    verbose=1,
    shuffle=True,
)

# ====================== 5. 模型评估函数 ======================
def evaluate_model(model, model_name, validation_generator, class_names):
    # 预测
    val_pred = model.predict(validation_generator)
    val_pred_classes = np.argmax(val_pred, axis=1)
    true_classes = validation_generator.classes
    
    # 分类报告
    print(f"\n=== {model_name} Classification Report ===")
    print(classification_report(true_classes, val_pred_classes, target_names=class_names, zero_division=0))
    
    # 混淆矩阵
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        confusion_matrix(true_classes, val_pred_classes),
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel('Predicted Pose')
    plt.ylabel('True Pose')
    plt.title(f'{model_name} Confusion Matrix')
    plt.show()
    
    return val_pred_classes

# ====================== 6. 全集验证 ======================
class_names = list(train_generator.class_indices.keys())  # 获取类别名

# 评估MobileNetV2
val_pred_mobilenet = evaluate_model(
    model_mobilenet, "MobileNetV2", validation_generator, class_names
)

# 评估ResNet50
val_pred_resnet = evaluate_model(
    model_resnet, "ResNet50", validation_generator, class_names
)

# ====================== 7. 训练曲线对比 ======================
plt.figure(figsize=(12, 8))

# 损失曲线对比
plt.subplot(2, 2, 1)
plt.plot(history_mobilenet.history['loss'], label='MobileNetV2 Training Loss')
plt.plot(history_mobilenet.history['val_loss'], label='MobileNetV2 Validation Loss')
plt.plot(history_resnet.history['loss'], label='ResNet50 Training Loss', linestyle='--')
plt.plot(history_resnet.history['val_loss'], label='ResNet50 Validation Loss', linestyle='--')
plt.title('Training Curves: Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss Value')
plt.legend()

# 准确率曲线对比
plt.subplot(2, 2, 2)
plt.plot(history_mobilenet.history['accuracy'], label='MobileNetV2 Training Acc')
plt.plot(history_mobilenet.history['val_accuracy'], label='MobileNetV2 Validation Acc')
plt.plot(history_resnet.history['accuracy'], label='ResNet50 Training Acc', linestyle='--')
plt.plot(history_resnet.history['val_accuracy'], label='ResNet50 Validation Acc', linestyle='--')
plt.title('Training Curves: Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

# ====================== 8. 单张验证（可视化推理过程，新增模型对比） ======================
# 随机选取3张验证图片（确保对比用相同图片）
random.seed(SEED)  # 固定随机种子，确保每次选图一致
validation_filepaths = validation_generator.filepaths
random_indices = random.sample(range(len(validation_filepaths)), 3)

def predict_and_visualize_comparison(image_path, true_label, model_list, model_names):
    # 预处理图片（仅需一次预处理，供多个模型使用）
    img = tf.keras.utils.load_img(image_path, target_size=IMAGE_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)  
    img_array = img_array / 255.0  # 标准化
    
    # 可视化图片
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"True Pose: {true_label}\nModel Predictions:")
    
    # 各模型预测结果
    predictions = []
    for model in model_list:
        probs = model.predict(img_array)
        predicted_class = class_names[np.argmax(probs)]
        confidence = np.max(probs).round(2)
        predictions.append((predicted_class, confidence))
    
    # 显示各模型结果
    for i, (name, (pred, conf)) in enumerate(zip(model_names, predictions)):
        plt.text(
            0.1, 0.9 - i*0.1,  # 坐标控制文字位置
            f"{name}: {pred} (Confidence: {conf})",
            bbox=dict(facecolor='white', alpha=0.8),
            transform=plt.gca().transAxes
        )
    
    plt.show()

# 对比验证
model_list = [model_mobilenet, model_resnet]
model_names = ["MobileNetV2", "ResNet50"]

for i in range(3):
    idx = random_indices[i]
    img_path = validation_filepaths[idx]
    true_label = class_names[validation_generator.labels[idx]]
    
    print(f"\n=== Image {i+1} Prediction Comparison ===")
    print(f"Image Path: {img_path}")
    predict_and_visualize_comparison(img_path, true_label, model_list, model_names)